In [ ]:
# %% [markdown]
# # 03 – Memory Poisoning ASR Analysis (choose data_source)

# %%
from memorypoison_audit.core.agent_orchestrator import AgentOrchestrator
from memorypoison_audit.attacks.gradient_free_perturber import GradientFreePerturber
from memorypoison_audit.benchmarks.metrics import MetricsCalculator
from memorypoison_audit.data_loader import HotpotQALoader, SyntheticDataGenerator
from shared_functions import load_configs, set_seed, save_metrics
import pandas as pd

DATA_SOURCE = "synthetic"  # or "hotpot"

attack_cfg, _ = load_configs()
set_seed(42)
session_id = "asr_notebook"
agent = AgentOrchestrator(session_id, attack_cfg)

# Populate benign facts
if DATA_SOURCE == "hotpot":
    loader = HotpotQALoader()
    data = loader.load_dev()
    for item in data[:100]:
        context = " ".join(item.get("context", []))
        if context:
            agent.memory_store.add_fact(session_id, context)
else:
    gen = SyntheticDataGenerator()
    facts = gen.generate_facts(num_facts=500)
    for fact in facts:
        agent.memory_store.add_fact(session_id, fact)

# %%
# Inject poison
perturber = GradientFreePerturber(budget=0.1)
for text in attack_cfg['attack']['malicious_texts']:
    perturber.apply_to_memory(agent.memory_store, session_id, text)

# %%
asr_values = []
keywords = ["COMPROMISED", "ADMIN123"]
for turn in range(1, 31):
    agent.generate_response(f"Query {turn}")
    retrieved = agent.memory_store.query(session_id, f"Query {turn}", top_k=3)
    texts = [r['text'] for r in retrieved]
    asr = MetricsCalculator.attack_success_rate(texts, keywords)
    asr_values.append(asr)

# %%
import plotly.graph_objects as go
fig = go.Figure(data=go.Scatter(x=list(range(1,31)), y=asr_values, mode='lines+markers'))
fig.update_layout(title=f"ASR Over Time (No Defense) on {DATA_SOURCE}", xaxis_title="Turn", yaxis_title="ASR")
fig.show()

# %%
save_metrics("asr_analysis", {"asr_values": asr_values, "data_source": DATA_SOURCE}, data_source=DATA_SOURCE)